# Clustering (KMeans) - Clientes de Atacado (UCI Wholesale Customers)
Análise de agrupamento (não usada nos dois exemplos originais, mas do mesmo tipo de análise de dados/ML).

Dataset: https://archive.ics.uci.edu/dataset/292/wholesale+customers
Baixe `Wholesale customers data.csv` e coloque na mesma pasta deste notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv("Wholesale customers data.csv")
df.head()

In [ ]:
# Estatísticas descritivas dos gastos anuais por categoria
df.describe().round(2)

In [ ]:
# Distribuição dos gastos por categoria (existem outliers fortes, comum em dados de varejo)
categorias = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
df[categorias].plot(kind="box", figsize=(10,5), showfliers=False)
plt.title("Gastos por Categoria (sem outliers extremos no gráfico)")
plt.ylabel("Gasto Anual")
plt.show()

In [ ]:
# Padroniza as variáveis antes do KMeans (escalas muito diferentes entre categorias)
X = df[categorias].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Método do cotovelo para escolher o número de clusters
inercias = []
K = range(1, 10)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

plt.plot(K, inercias, marker="o")
plt.xlabel("Número de Clusters (k)")
plt.ylabel("Inércia")
plt.title("Método do Cotovelo")
plt.show()

In [ ]:
# Treina o KMeans com k=4 (ajustável conforme o cotovelo acima)
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print("Silhouette Score:", silhouette_score(X_scaled, df["cluster"]))
df["cluster"].value_counts()

In [ ]:
# Reduz para 2D com PCA apenas para visualizar os clusters
pca = PCA(n_components=2)
componentes = pca.fit_transform(X_scaled)
df["pca1"], df["pca2"] = componentes[:,0], componentes[:,1]

sns.scatterplot(x="pca1", y="pca2", hue="cluster", palette="Set2", data=df)
plt.title("Clusters de Clientes (visualização em 2D via PCA)")
plt.show()

In [ ]:
# Perfil médio de gastos por cluster (interpretação dos grupos)
df.groupby("cluster")[categorias].mean().round(1)